### Step 1: Install Dependencies

In [20]:
!pip install -U langsmith langchain-openai langchain-community openai openevals PyMuPDF pandas

  Using cached pandas-2.3.3-cp312-cp312-macosx_11_0_arm64.whl.metadata (91 kB)
  Using cached pytz-2025.2-py2.py3-none-any.whl.metadata (22 kB)
  Using cached tzdata-2025.2-py2.py3-none-any.whl.metadata (1.4 kB)
Using cached pandas-2.3.3-cp312-cp312-macosx_11_0_arm64.whl (10.7 MB)
Using cached pytz-2025.2-py2.py3-none-any.whl (509 kB)
Using cached tzdata-2025.2-py2.py3-none-any.whl (347 kB)

[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [2]:
import os
import getpass

# Set up environment variables with your input
print("Please enter your API keys to get started:")
print("=" * 50)

# LangSmith tracing setting
langsmith_tracing = input("Enable LangSmith tracing? (true/false) [default: true]: ").strip() or "true"
os.environ["LANGSMITH_TRACING"] = langsmith_tracing

# LangSmith API key (secure input)
if not os.getenv("LANGSMITH_API_KEY"):
    langsmith_api_key = getpass.getpass("Enter your LangSmith API key: ")
    os.environ["LANGSMITH_API_KEY"] = langsmith_api_key
else:
    print("✓ LangSmith API key already set")

# OpenAI API key (secure input)  
if not os.getenv("OPENAI_API_KEY"):
    openai_api_key = getpass.getpass("Enter your OpenAI API key: ")
    os.environ["OPENAI_API_KEY"] = openai_api_key
else:
    print("✓ OpenAI API key already set")

print("\n✓ Environment setup complete!")
print("You can now proceed with the rest of the notebook.")

Please enter your API keys to get started:
✓ LangSmith API key already set
✓ OpenAI API key already set

✓ Environment setup complete!
You can now proceed with the rest of the notebook.


In [3]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

### Step 2: Create Vector Database

We'll build a basic RAG application using your internal documents. Our simple implementation follows three core steps:

### Internal Data Focus
* **Source material:** Internal documents

### Key Components
* **Indexing:** Chunk and embed internal documents into a vector store
* **Retrieval:** Find relevant document chunks based on financial questions  
* **Generation:** Combine retrieved context with user questions for LLM processing

In [4]:
# List of PDF files to load
pdf_files = [
    "ASSET_PURCHASE_AGREEMENT_CHALLENGE.pdf"
]

# Load documents from the URLs
docs = [PyMuPDFLoader(pdf_file).load() for pdf_file in pdf_files]
docs_list = [item for sublist in docs for item in sublist]

# Initialize a text splitter with specified chunk size and overlap
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=500, chunk_overlap=0
)

# Split the documents into chunks
doc_splits = text_splitter.split_documents(docs_list)

# Add the document chunks to the "vector store" using OpenAIEmbeddings
vectorstore = InMemoryVectorStore.from_documents(
    documents=doc_splits,
    embedding=OpenAIEmbeddings(),
)

In [5]:
# With langchain we can easily turn any vector store into a retrieval component:
retriever = vectorstore.as_retriever(k=6)

In [6]:
# Use the Retriever with an Example Question

# Example question about the Asset Purchase Agreement
question = "What is the purchase price in this agreement?"

# Use the retriever to find relevant document chunks
retrieved_docs = retriever.invoke(question)

# Display the retrieved documents
print(f"Question: {question}\n")
print(f"Retrieved {len(retrieved_docs)} relevant document chunks:\n")
print("=" * 80)

for i, doc in enumerate(retrieved_docs, 1):
    print(f"\n📄 Document Chunk {i}:")
    print(f"Content: {doc.page_content[:200]}...")  # Show first 200 chars
    print(f"Metadata: {doc.metadata}")
    print("-" * 80)


/Users/rickchakra/Projects/aderant-build-ai-workshop/venv/lib/python3.12/site-packages/pydantic/v1/main.py:1054: UserWarning: LangSmith now uses UUID v7 for run and trace identifiers. This warning appears when passing custom IDs. Please use: from langsmith import uuid7
            id = uuid7()
Future versions will require UUID v7.
  input_data = validator(cls_, input_data)


Question: What is the purchase price in this agreement?

Retrieved 4 relevant document chunks:


📄 Document Chunk 1:
Content: ASSET PURCHASE AGREEMENT 
 
 
This Asset Purchase Agreement (this "Agreement") is entered into as of November 8, 2024 
(the "EGective Date"), by and between: 
 
GlobalTech Ventures LLC, a Delaware lim...
Metadata: {'producer': 'macOS Version 15.5 (Build 24F74) Quartz PDFContext', 'creator': '', 'creationdate': "D:20251118221635Z00'00'", 'source': 'ASSET_PURCHASE_AGREEMENT_CHALLENGE.pdf', 'file_path': 'ASSET_PURCHASE_AGREEMENT_CHALLENGE.pdf', 'total_pages': 27, 'format': 'PDF 1.3', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': "D:20251118221635Z00'00'", 'trapped': '', 'modDate': "D:20251118221635Z00'00'", 'creationDate': "D:20251118221635Z00'00'", 'page': 0}
--------------------------------------------------------------------------------

📄 Document Chunk 2:
Content: 2.1 Purchase and Sale of Assets. Subject to the terms and conditions of th

In [7]:
from langchain_openai import ChatOpenAI
from langsmith import traceable

llm = ChatOpenAI(model="gpt-5.1")

# Add decorator so this function is traced in LangSmith
@traceable()
def rag_bot(question: str) -> dict:
    # LangChain retriever will be automatically traced
    docs = retriever.invoke(question)
    docs_string = "".join(doc.page_content for doc in docs)

    instructions = f"""You are a legal analyst who is an expert at analyzing legal contracts and answering questions.

    Documents:
    {docs_string}"""

    # langchain ChatModel will be automatically traced
    ai_msg = llm.invoke([
            {"role": "system", "content": instructions},
            {"role": "user", "content": question},
        ],
    )

    return {"answer": ai_msg.content, "documents": docs}

In [8]:
# Simple example: Call rag_bot with a question
result = rag_bot("Who are the parties?")

# Display the answer
print("Question: Who are the parties?\n")
print("=" * 80)
print(f"\nAnswer: {result['answer']}\n")
print("=" * 80)
print(f"\nNumber of documents retrieved: {len(result['documents'])}")


Question: Who are the parties?


Answer: The parties to this Agreement are:

1. **GLOBALTECH VENTURES LLC**  
2. **INNOVATESOFT CORPORATION**


Number of documents retrieved: 4


## Create Eval Dataset

In [9]:
from langsmith import Client

client = Client()

# Define the examples for the dataset
examples = [
    {
        "inputs": {"question": "What is the base purchase price for the acquisition of InnovateSoft Corporation?"},
        "outputs": {"answer": "The base purchase price is $42,000,000 (Forty-Two Million Dollars)"},
    },
    {
        "inputs": {"question": "What is the maximum total consideration including all earnout payments that Seller could receive?"},
        "outputs": {"answer": "The maximum total consideration is $55,500,000, consisting of the $42,000,000 base purchase price plus up to $13,500,000 in earnout payments over the two-year earnout period"},
    },
    {
        "inputs": {"question": "What is the indemnification cap for breaches of Intellectual Property representations, and how does it compare to the basket amount and the general indemnification cap?"},
        "outputs": {"answer": "The indemnification cap for IP breaches is $15,000,000, which is 5 times higher than the general breach cap of $3,000,000 (the Escrow Amount) and 30 times higher than the $500,000 basket amount. Notably, IP breaches are also exempt from the basket amount requirement, meaning Buyer can claim from the first dollar of loss"},
    }
]

# Create the dataset and examples in LangSmith
dataset_name = "Legal Contract RAG"
dataset = client.create_dataset(dataset_name=dataset_name)
client.create_examples(
    dataset_id=dataset.id,
    examples=examples
)

{'example_ids': ['ab6e42bc-48f6-457e-bde4-ba8e3d34e87a',
  '553be277-c06b-49d4-b81a-c89d035e3294',
  '21b0289f-bd35-47ff-afd9-6e82796708e0'],
 'count': 3}

## RAG Evaluation Framework

We'll evaluate your financial RAG system using four key dimensions. Each evaluator compares different components of the RAG pipeline:

### 1. Correctness: Response vs Reference Answer
* **Goal:** How accurate is the financial analysis compared to expected answers?
* **Requirements:** Ground truth answers in your dataset
* **Method:** LLM-as-judge assesses factual accuracy of financial information
* **Focus:** Validates correct financial metrics, trends, and company insights

### 2. Relevance: Response vs Input Question  
* **Goal:** How well does the answer address the original financial question?
* **Requirements:** No reference answer needed
* **Method:** LLM-as-judge evaluates response helpfulness and directness
* **Focus:** Ensures answers stay on-topic for financial queries

### 3. Groundedness: Response vs Retrieved Documents
* **Goal:** How faithful is the response to the source financial documents?
* **Requirements:** No reference answer needed  
* **Method:** LLM-as-judge detects hallucinations and unsupported claims
* **Focus:** Prevents fabricated financial data or analysis

### 4. Retrieval Relevance: Retrieved Docs vs Input Question
* **Goal:** How relevant are the retrieved documents for answering the question?
* **Requirements:** No reference answer needed
* **Method:** LLM-as-judge assesses document-question alignment  
* **Focus:** Validates that the right financial reports were found

### Evaluation Strategy
Each metric provides unique insights into your RAG system's performance, helping identify whether issues stem from retrieval, generation, or both components.

### Diagram
https://docs.smith.langchain.com/assets/images/rag_eval_overview-0d95d78db4d60c2bccbd333f8ba75e60.png

In [10]:
from typing_extensions import Annotated, TypedDict

# Grade output schema
class CorrectnessGrade(TypedDict):
    # Note that the order in the fields are defined is the order in which the model will generate them.
    # It is useful to put explanations before responses because it forces the model to think through
    # its final response before generating it:
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    correct: Annotated[bool, ..., "True if the answer is correct, False otherwise."]

# Grade prompt
correctness_instructions = """You are an expert data labeler evaluating model outputs for correctness.

You will be given a QUESTION, the GROUND TRUTH (correct) ANSWER, and the MODEL ANSWER. 

Here is the grade criteria to follow:
(1) Grade the model answers based ONLY on their factual accuracy relative to the ground truth answer. 
(2) Ensure that the model answer does not contain any conflicting statements.
(3) It is OK if the model answer contains more information than the ground truth answer, as long as it is factually accurate relative to the  ground truth answer.

Correctness:
A correctness value of True means that the model's answer meets all of the criteria.
A correctness value of False means that the model's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

# Grader LLM
grader_llm = ChatOpenAI(model="gpt-5.1", temperature=0).with_structured_output(CorrectnessGrade, method="json_schema", strict=True)

def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    """An evaluator for RAG answer accuracy"""
    answers = f"""\
QUESTION: {inputs['question']}
GROUND TRUTH ANSWER: {reference_outputs['answer']}
MODEL ANSWER: {outputs['answer']}"""

    # Run evaluator
    grade = grader_llm.invoke([
        {"role": "system", "content": correctness_instructions}, 
        {"role": "user", "content": answers}
    ])
    return grade["correct"]

In [11]:
# Grade output schema
class RelevanceGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    relevant: Annotated[bool, ..., "Provide the score on whether the answer addresses the question"]

# Grade prompt
relevance_instructions="""You are an expert data labeler evaluating model outputs for correctness.

You will be given a QUESTION and a MODEL ANSWER. 

Here is the grade criteria to follow:
(1) Ensure the MODEL ANSWER is concise and relevant to the QUESTION
(2) Ensure the MODEL ANSWER helps to answer the QUESTION

Relevance:
A relevance value of True means that the model's answer meets all of the criteria.
A relevance value of False means that the model's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

# Grader LLM
relevance_llm = ChatOpenAI(model="gpt-5.1", temperature=0).with_structured_output(RelevanceGrade, method="json_schema", strict=True)

# Evaluator
def relevance(inputs: dict, outputs: dict) -> bool:
    """A simple evaluator for RAG answer helpfulness."""
    answer = f"QUESTION: {inputs['question']}\nMODEL ANSWER: {outputs['answer']}"
    grade = relevance_llm.invoke([
        {"role": "system", "content": relevance_instructions}, 
        {"role": "user", "content": answer}
    ])
    return grade["relevant"]

In [12]:
# Grade output schema
class GroundedGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    grounded: Annotated[bool, ..., "Provide the score on if the answer hallucinates from the documents"]

# Grade prompt
grounded_instructions = """You are an expert data labeler evaluating model outputs for correctness. 

You will be given FACTS and a MODEL ANSWER. 

Here is the grade criteria to follow:
(1) Ensure the MODEL ANSWER is grounded in the FACTS. 
(2) Ensure the MODEL ANSWER does not contain "hallucinated" information outside the scope of the FACTS.

Grounded:
A grounded value of True means that the student's answer meets all of the criteria.
A grounded value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

# Grader LLM 
grounded_llm = ChatOpenAI(model="gpt-5.1", temperature=0).with_structured_output(GroundedGrade, method="json_schema", strict=True)

# Evaluator
def groundedness(inputs: dict, outputs: dict) -> bool:
    """A simple evaluator for RAG answer groundedness."""
    doc_string = "\n\n".join(doc.page_content for doc in outputs["documents"])
    answer = f"FACTS: {doc_string}\nMODEL ANSWER: {outputs['answer']}"
    grade = grounded_llm.invoke([{"role": "system", "content": grounded_instructions}, {"role": "user", "content": answer}])
    return grade["grounded"]

In [13]:
# Grade output schema
class RetrievalRelevanceGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    relevant: Annotated[bool, ..., "True if the retrieved documents are relevant to the question, False otherwise"]

# Grade prompt
retrieval_relevance_instructions = """You are an expert data labeler evaluating information retrieval (RAG) outputs for correctness. 

You will be given a QUESTION and a set of FACTS provided by the RAG MODEL. 

Here is the grade criteria to follow:
(1) You goal is to identify FACTS that are completely unrelated to the QUESTION
(2) If the facts contain ANY keywords or semantic meaning related to the question, consider them relevant
(3) It is OK if the facts have SOME information that is unrelated to the question as long as (2) is met

Relevance:
A relevance value of True means that the FACTS contain ANY keywords or semantic meaning related to the QUESTION and are therefore relevant.
A relevance value of False means that the FACTS are completely unrelated to the QUESTION.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

# Grader LLM
retrieval_relevance_llm = ChatOpenAI(model="gpt-5.1", temperature=0).with_structured_output(RetrievalRelevanceGrade, method="json_schema", strict=True)

def retrieval_relevance(inputs: dict, outputs: dict) -> bool:
    """An evaluator for document relevance"""
    doc_string = "\n\n".join(doc.page_content for doc in outputs["documents"])
    answer = f"FACTS: {doc_string}\nQUESTION: {inputs['question']}"

    # Run evaluator
    grade = retrieval_relevance_llm.invoke([
        {"role": "system", "content": retrieval_relevance_instructions}, 
        {"role": "user", "content": answer}
    ])
    return grade["relevant"]

In [17]:
import tqdm as notebook_tqdm

In [18]:
def target(inputs: dict) -> dict:
    return rag_bot(inputs["question"])

experiment_results = client.evaluate(
    target,
    data=dataset_name,
    evaluators=[correctness, groundedness, relevance, retrieval_relevance],
    experiment_prefix="rag-doc-relevance",
    metadata={"version": "gpt-5.1"},
)


View the evaluation results for experiment: 'rag-doc-relevance-63d5c925' at:
https://smith.langchain.com/o/b8c00c25-9820-5aa4-a030-cfcde0aef6cf/datasets/5bb59cc7-7d15-483e-b14a-d18f0de8db19/compare?selectedSessions=5c7ba166-befe-4364-8290-e45dea4e8311




3it [00:46, 15.57s/it]


In [21]:
import pandas as pd

# Explore results locally as a dataframe if you have pandas installed
experiment_results.to_pandas()

,inputs.question,outputs.answer,outputs.documents,error,reference.answer,feedback.correctness,feedback.groundedness,feedback.relevance,feedback.retrieval_relevance,execution_time,example_id,id
0,What is the indemnification cap for breaches o...,- **IP indemnification cap (Section 4.1(f) – I...,[page_content='aggregate amount of all Losses ...,None,The indemnification cap for IP breaches is $15...,True,True,True,True,6.422959,21b0289f-bd35-47ff-afd9-6e82796708e0,019a9c48-2127-705a-a6ea-1559a36301e6
1,What is the maximum total consideration includ...,Based on the provisions you’ve provided:\n\n1....,"[page_content='accordance with GAAP, consisten...",None,"The maximum total consideration is $55,500,000...",False,True,True,True,4.943519,553be277-c06b-49d4-b81a-c89d035e3294,019a9c48-7425-759d-adab-7b19ad54c85e
2,What is the base purchase price for the acquis...,The base purchase price for the acquisition of...,[page_content='ASSET PURCHASE AGREEMENT \n \n ...,None,"The base purchase price is $42,000,000 (Forty-...",True,True,True,True,0.925154,ab6e42bc-48f6-457e-bde4-ba8e3d34e87a,019a9c48-bc37-7598-8b73-286373b75c67
